# Chapter 14 — The GPU: Memory, Tensor Cores, and the Roofline

Every number in Chapter 14 is arithmetic on a handful of hardware figures, so this notebook
recomputes all of them from those figures and **asserts** each printed value. There is no GPU here:
Exercises 14.2 and 14.4 are measured on this machine's CPU, which shows the same shapes (a rising
then flat FLOP/s curve; strided access costing more than contiguous) at CPU scale.

In [ ]:
import math, sys, os
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)
H100 = dict(flops=989.4e12, bw=3.35e12)
def roofline(flops, bytes_moved, peak_flops=989.4e12, bw=3.35e12):
    t_compute, t_memory = flops / peak_flops, bytes_moved / bw
    return dict(ai=flops / bytes_moved,
                bound="compute" if t_compute > t_memory else "memory",
                seconds=max(t_compute, t_memory), t_compute=t_compute, t_memory=t_memory,
                frac_of_peak=min(1.0, (flops / bytes_moved) / (peak_flops / bw)))

## 14.1 — the ridge point, and how it moved across four chips

In [ ]:
ridge = H100["flops"] / H100["bw"]
print(f"H100 ridge: {ridge:.1f} FLOP/byte")
assert round(ridge) == 295
chips = {"A100 80GB": (312, 2.04, 153), "H100 SXM5": (989, 3.35, 295), "H200 SXM": (989, 4.80, 206),
         "B200": (2250, 8.00, 281)}
for name, (tf, tb, want) in chips.items():
    print(f"{name:10} {tf / tb:6.1f}")
    assert round(tf / tb) == want

## 14.2 — a square bf16 matmul has AI = n/3; the crossover is n ≈ 886

In [ ]:
def matmul_ai(M, K, N, bytes_per=2):
    return 2 * M * K * N / (bytes_per * (M * K + K * N + M * N))
table = {128: 42.7, 256: 85.3, 512: 170.7, 1024: 341.3, 4096: 1365.3}
for n, want in table.items():
    ai = matmul_ai(n, n, n)
    verdict = "COMPUTE" if ai > ridge else "MEMORY"
    print(f"n={n:5d}  AI={ai:7.1f}  {verdict}-bound")
    assert round(ai, 1) == want and abs(ai - n / 3) < 1e-9
    assert (verdict == "COMPUTE") == (n >= 1024)
assert round(3 * ridge) == 886
assert round(100 * matmul_ai(512, 512, 512) / ridge) == 58          # 171/295 = 58% of peak

## 14.3 — one decoded token of a 7B bf16 model: memory-bound by ~300×

In [ ]:
P = 7e9
r = roofline(2 * P, 2 * P)                       # 14 GFLOP, 14 GB
print(f"AI {r['ai']:.0f}; memory {r['t_memory'] * 1e3:.3f} ms; compute {r['t_compute'] * 1e3:.3f} ms; "
      f"{r['t_memory'] / r['t_compute']:.0f}x; ceiling {1 / r['t_memory']:.0f} tok/s; idle {1 - 1 / ridge:.1%}")
assert r["ai"] == 1 and r["bound"] == "memory"
assert round(r["t_memory"] * 1e3, 3) == 4.179 and round(r["t_compute"] * 1e3, 3) == 0.014
assert round(r["t_memory"] / r["t_compute"], -1) == 300 and round(1 / r["t_memory"]) == 239
assert round(100 * (1 - 1 / ridge), 1) == 99.7
pre = roofline(2 * P * 2048, 2 * P)
print(f"prefill 2048: AI {pre['ai']:.0f}; memory {pre['t_memory'] * 1e3:.3f} ms; compute {pre['t_compute'] * 1e3:.3f} ms")
assert pre["ai"] == 2048 and pre["bound"] == "compute" and round(pre["t_compute"] * 1e3, 3) == 28.979

## 14.3 — decode batching: AI = B, and the fraction of peak it can reach

In [ ]:
want = {1: 0.3, 8: 2.7, 32: 10.8, 128: 43.3, 256: 86.7, 295: 99.9}
for B, pct in want.items():
    r = roofline(2 * P * B, 2 * P)
    print(f"batch {B:3d}: AI {r['ai']:.0f}  {100 * r['frac_of_peak']:5.1f}% of peak")
    assert r["ai"] == B and round(100 * r["frac_of_peak"], 1) == pct

## 14.4 — tensor-core alignment: vocab padding and head sizes

In [ ]:
assert 50304 % 64 == 0 and 50304 == 64 * math.ceil(50257 / 64)          # nanoGPT's padded vocab
assert 4096 // 32 == 128 and 768 // 12 == 64 and 2560 // 32 == 80
for d_head in (64, 80, 96, 128, 192, 256):
    assert d_head % 16 == 0                                            # all tensor-core friendly
assert 80 == 5 * 16

## 14.5 — coalescing: how many 128-byte lines one warp's load touches

32 threads each read one 4-byte word at `stride × thread` (in words).

In [ ]:
def lines_per_warp(stride, word=4, line=128):
    return len({(t * stride * word) // line for t in range(32)})
for s, want in ((1, 1), (2, 2), (4, 4), (32, 32)):
    print(f"stride {s:2d}: {lines_per_warp(s)} lines, {100 / lines_per_warp(s):.0f}% of fetched bytes used")
    assert lines_per_warp(s) == want
assert 32 * 4 == 128

## 14.6 — fusion, and the T×T score matrix FlashAttention never writes

**Resolved (November 2026).** This cell found an error in the chapter, and the chapter text was corrected to match the measurement below. The original finding: §14.6 calls shared memory "~7× the bandwidth of HBM"; with the table's own
figures (~20 TB/s vs 3.35 TB/s) that is **~6×**. The "128 MB per head" for T = 8192 is in MiB
(134 MB in decimal units) — consistent with the course's other figures, just binary.

In [ ]:
unfused_trips, fused_trips = 3, 1
assert unfused_trips / fused_trips == 3                             # same FLOPs, 3x the intensity
S = 8192 * 8192 * 2                                                  # bf16 scores, one head
print(f"T=8192 scores: {S / 2**20:.0f} MiB = {S / 1e6:.0f} MB, vs a 50 MB L2; SRAM/HBM = {20 / 3.35:.1f}x")
assert S == 128 * 2**20 and S / 1e6 > 50
assert round(20 / 3.35) == 6                                         # chapter: ~7x

## Exercise 14.1 — the roofline calculator on one decode step of a d = 4096 block (B = T = 1)

bf16 weights and activations; SwiGLU with d_ff = 11008 (Llama-2-7B), MHA projections 4096×4096.
Element-wise ops do a few FLOPs per element. Everything is memory-bound, and the time is almost
entirely the seven weight matrices; the norms, residual adds and softmax are a rounding error.

In [ ]:
d, ff = 4096, 11008
ops = {f"attn {n}": (2 * d * d, 2 * d * d + 4 * d) for n in ("q", "k", "v", "o")}
ops.update({f"swiglu {n}": (2 * d * ff, 2 * d * ff + 2 * (d + ff)) for n in ("gate", "up", "down")})
ops["rmsnorm x2"] = (2 * 4 * d, 2 * 3 * 2 * d)
ops["residual add x2"] = (2 * d, 2 * 3 * 2 * d)
ops["softmax (ctx 1)"] = (5, 2 * 2)
res = {k: roofline(f, b) for k, (f, b) in ops.items()}
for k, r in sorted(res.items(), key=lambda kv: -kv[1]["seconds"]):
    print(f"{k:16} AI {r['ai']:6.2f}  {r['bound']:7}  {r['seconds'] * 1e6:8.3f} us")
assert all(r["bound"] == "memory" for r in res.values())
total = sum(r["seconds"] for r in res.values())
tiny_ops = sum(res[k]["seconds"] for k in ("rmsnorm x2", "residual add x2", "softmax (ctx 1)"))
print(f"block decode step {total * 1e6:.1f} us; norm/add/softmax share {tiny_ops / total:.3%}")
assert tiny_ops / total < 0.01

## Exercise 14.2 — find this CPU's ridge empirically

Achieved FLOP/s rises with n and then flattens. On this CPU the knee is at a few hundred — far below
a GPU's ~886, because a CPU's compute-to-bandwidth ratio is much smaller.

In [ ]:
import time
torch.set_num_threads(min(4, torch.get_num_threads()))
achieved = {}
for n in (64, 128, 256, 512, 1024):
    a, b = torch.randn(n, n), torch.randn(n, n)
    for _ in range(3): a @ b
    reps = max(3, int(1e8 / n ** 3))
    t = time.perf_counter()
    for _ in range(reps): a @ b
    achieved[n] = 2 * n ** 3 / ((time.perf_counter() - t) / reps)
    print(f"n={n:5d}  {achieved[n] / 1e9:7.1f} GFLOP/s")
assert achieved[1024] > 1.5 * achieved[64]                        # rises (loose: timing on a busy machine)
assert achieved[1024] < 3 * achieved[512]                         # ... then flattens

## Exercise 14.3 — 70B on 8×H100: does the compute-bound decode batch fit?

Assumes Llama-70B's 80 layers (the exercise gives 8 KV heads and d_head = 128) and bf16 cache.

In [ ]:
agg_flops, agg_bw = 7912e12, 26.8e12
B_star = agg_flops / agg_bw
kv_per_seq = 2 * 80 * 8 * 128 * 2 * 8192                           # K and V, layers, heads, d, bytes, ctx
free = 8 * 80e9 - 140e9
fits = free / kv_per_seq
print(f"compute-bound at B ~ {B_star:.0f}; KV/seq {kv_per_seq / 1e9:.2f} GB; "
      f"{free / 1e9:.0f} GB free holds {fits:.0f} sequences")
assert round(B_star) == 295 and round(kv_per_seq / 1e9, 2) == 2.68
assert fits < B_star                                                # it does not fit: hence GQA/MLA/paging

## Exercise 14.4 — the cost of a bad layout (CPU)

Summing every 16th float of a 256 MB array versus a contiguous run of the same number of floats.
Same useful bytes; the strided read pulls a whole cache line for every element it uses.

In [ ]:
x = torch.randn(8192 * 8192)
def bench(fn, reps=5):
    fn(); t = time.perf_counter()
    for _ in range(reps): fn()
    return (time.perf_counter() - t) / reps
n = x.numel() // 16
assert x[::16].numel() == x[:n].numel()
t_strided, t_contig = bench(lambda: x[::16].sum()), bench(lambda: x[:n].sum())
print(f"strided {t_strided * 1e3:.2f} ms, contiguous {t_contig * 1e3:.2f} ms: {t_strided / t_contig:.1f}x")
assert t_strided > 1.5 * t_contig                                  # measured 3.5-7.7x on this CPU